In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/ADNI_Project/Early_Dementia_Risk_Longitudinal_Experiment')
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COHORT_PATH = DATA_DIR / 'longitudinal_mci_36m.csv'
RANDOM_STATE = 42
print('Project folder:', PROJECT_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

cog = pd.read_csv(RESULTS_DIR / 'cognitive_predictions.csv').rename(columns={'PROBABILITY': 'COG_PROB'})
mri = pd.read_csv(RESULTS_DIR / 'mri_predictions.csv').rename(columns={'PROBABILITY': 'MRI_PROB'})
merged = cog[['RID', 'SPLIT', 'OUTCOME', 'COG_PROB']].merge(
    mri[['RID', 'SPLIT', 'OUTCOME', 'MRI_PROB']],
    on=['RID', 'SPLIT', 'OUTCOME'], validate='one_to_one'
)
val = merged[merged.SPLIT == 'validation'].copy()
test = merged[merged.SPLIT == 'test'].copy()
print('Validation:', len(val), 'Test:', len(test))


In [ ]:
weights = np.linspace(0, 1, 101)  # weight applied to cognitive probability
search_rows = []
for weight in weights:
    prob = weight * val['COG_PROB'] + (1 - weight) * val['MRI_PROB']
    search_rows.append({'cognitive_weight': weight, 'validation_roc_auc': roc_auc_score(val['OUTCOME'], prob)})

weight_search = pd.DataFrame(search_rows)
best_row = weight_search.sort_values(['validation_roc_auc', 'cognitive_weight'], ascending=[False, True]).iloc[0]
best_weight = float(best_row['cognitive_weight'])
val_prob = best_weight * val['COG_PROB'] + (1 - best_weight) * val['MRI_PROB']

fpr, tpr, thresholds = roc_curve(val['OUTCOME'], val_prob)
valid = np.isfinite(thresholds)
threshold = float(thresholds[valid][np.argmax((tpr - fpr)[valid])])

test_prob = best_weight * test['COG_PROB'] + (1 - best_weight) * test['MRI_PROB']
test_pred = (test_prob >= threshold).astype(int)

print('Selected cognitive weight:', round(best_weight, 2))
print('Selected MRI weight:', round(1 - best_weight, 2))
print('Validation-selected threshold:', round(threshold, 3))
print('Test ROC-AUC:', round(roc_auc_score(test['OUTCOME'], test_prob), 3))
print(classification_report(test['OUTCOME'], test_pred, target_names=['stable MCI', 'progressive MCI']))


In [ ]:
output = pd.concat([
    val[['RID', 'SPLIT', 'OUTCOME']].assign(PROBABILITY=val_prob),
    test[['RID', 'SPLIT', 'OUTCOME']].assign(PROBABILITY=test_prob),
], ignore_index=True)
output['PREDICTION'] = (output['PROBABILITY'] >= threshold).astype(int)
output.to_csv(RESULTS_DIR / 'multimodal_predictions.csv', index=False)
weight_search.to_csv(RESULTS_DIR / 'fusion_weight_search.csv', index=False)
(RESULTS_DIR / 'multimodal_model_info.json').write_text(json.dumps({
    'fusion_method': 'validation-selected weighted average',
    'cognitive_weight': best_weight,
    'mri_weight': 1 - best_weight,
    'threshold': threshold,
}, indent=2))

cm = confusion_matrix(test['OUTCOME'], test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['stable', 'progressive'], yticklabels=['stable', 'progressive'])
plt.title('Multimodal test confusion matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'multimodal_confusion_matrix.png', dpi=200)
plt.show()
